<a href="https://colab.research.google.com/github/soltsega/multilingual_ner/blob/main/notebooks/ner_full_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NER Project Pipeline - CoNLL-2003 + MasakhaNER


# Multilingual KYC Named Entity Recognition — Model Development

## Introduction

This notebook presents the end-to-end development of a **multilingual Named Entity Recognition (NER) system for KYC-oriented text**, developed as part of the **Ethiopian Information Network Security Agency (INSA) Summer Camp**.

The primary objective is to develop a transformer-based NLP model capable of identifying and classifying identity-related entities from multilingual text. The notebook covers the complete machine-learning workflow, beginning with dataset exploration and preparation and continuing through label standardization, dataset merging, tokenization, model fine-tuning, and evaluation.

Because the project combines multiple annotated datasets, particular attention is given to **dataset compatibility and label alignment**. The datasets may differ in their label names, annotation conventions, and distributions. Therefore, the preprocessing pipeline standardizes the annotations into a consistent representation before training.

The notebook uses the **BIO (Beginning–Inside–Outside) labeling scheme** to represent entity boundaries and prepares the data for transformer-based token classification.

The overall workflow is:

```text
Dataset Collection
       ↓
Dataset Exploration
       ↓
Data Cleaning & Validation
       ↓
Label Standardization
       ↓
Dataset Merging
       ↓
BIO Label Preparation
       ↓
Train / Validation / Test Split
       ↓
Tokenization & Label Alignment
       ↓
Transformer Fine-Tuning
       ↓
Evaluation
       ↓
Model Export
       ↓
Hugging Face Deployment

# Install dependencies

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

MODEL_DIR = "/content/drive/MyDrive/NER_Project/ner-multilingual-model"

os.makedirs(MODEL_DIR, exist_ok=True)

print("Model will be saved to:")
print(MODEL_DIR)

Model will be saved to:
/content/drive/MyDrive/NER_Project/ner-multilingual-model


In [ ]:
# Cell 1 — Install dependencies

%pip install -q -U \
    datasets \
    transformers \
    seqeval \
    accelerate \
    evaluate \
    sentencepiece \
    safetensors

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 24.5 MB/s eta 0:00:00


In [ ]:
import torch
import transformers
import datasets
import evaluate

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
Transformers: 5.15.0
Datasets: 5.0.1
CUDA available: True
GPU: Tesla T4


# Load tokenizer and datasets

In [ ]:
from transformers import AutoTokenizer
from datasets import load_dataset

MODEL_CHECKPOINT = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_CHECKPOINT,
    use_fast=True
)

print(f"Tokenizer loaded: {MODEL_CHECKPOINT}")

print("\nLoading CoNLL-2003...")

conll = load_dataset("tomaarsen/conll2003")

print("CoNLL-2003 loaded successfully.")
print(f"Train:      {len(conll['train'])}")
print(f"Validation: {len(conll['validation'])}")
print(f"Test:       {len(conll['test'])}")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Tokenizer loaded: xlm-roberta-base

Loading CoNLL-2003...


...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

CoNLL-2003 loaded successfully.
Train:      14041
Validation: 3250
Test:       3453


In [ ]:
# ============================================================
# Load MasakhaNER 1.0
# ============================================================

import requests
from datasets import Dataset, DatasetDict


# ------------------------------------------------------------
# MasakhaNER 1.0 languages
# ------------------------------------------------------------

masakha_langs = [
    "amh",  # Amharic
    "hau",  # Hausa
    "ibo",  # Igbo
    "kin",  # Kinyarwanda
    "lug",  # Luganda
    "luo",  # Luo
    "pcm",  # Nigerian Pidgin
    "swa",  # Swahili
    "wol",  # Wolof
    "yor"   # Yoruba
]


# ------------------------------------------------------------
# GitHub repository containing MasakhaNER 1.0
# ------------------------------------------------------------

BASE_URL = (
    "https://raw.githubusercontent.com/"
    "masakhane-io/masakhane-ner/main/data"
)


# ------------------------------------------------------------
# Read CoNLL file
# ------------------------------------------------------------

def read_conll(url):
    """
    Read a MasakhaNER 1.0 CoNLL file.

    Each non-empty line contains:
        TOKEN<TAB>LABEL

    Sentences are separated by blank lines.
    """

    response = requests.get(url)

    response.raise_for_status()

    lines = response.text.splitlines()

    tokens = []
    ner_tags = []

    examples = []

    for line in lines:

        line = line.strip()

        # Blank line = end of sentence
        if not line:

            if tokens:

                examples.append({
                    "tokens": tokens,
                    "ner_tags": ner_tags
                })

                tokens = []
                ner_tags = []

            continue

        # CoNLL format:
        # token<TAB>tag
        parts = line.split()

        if len(parts) < 2:
            continue

        token = parts[0]
        tag = parts[-1]

        tokens.append(token)
        ner_tags.append(tag)

    # Handle final sentence
    if tokens:

        examples.append({
            "tokens": tokens,
            "ner_tags": ner_tags
        })

    return examples


# ------------------------------------------------------------
# BIO label mapping
# ------------------------------------------------------------

label_list = [
    "O",
    "B-PER",
    "I-PER",
    "B-ORG",
    "I-ORG",
    "B-LOC",
    "I-LOC",
    "B-DATE",
    "I-DATE"
]

label2id = {
    label: i
    for i, label in enumerate(label_list)
}


# ------------------------------------------------------------
# Load each language
# ------------------------------------------------------------

masakha_datasets = {}

for lang in masakha_langs:

    print(f"\nLoading MasakhaNER 1.0: {lang}")

    splits = {}

    for split in ["train", "dev", "test"]:

        url = (
            f"{BASE_URL}/{lang}/"
            f"{split}.txt"
        )

        examples = read_conll(url)

        # Convert string labels → integer IDs
        for example in examples:

            example["ner_tags"] = [
                label2id[tag]
                for tag in example["ner_tags"]
            ]

        # MasakhaNER calls validation "dev".
        split_name = (
            "validation"
            if split == "dev"
            else split
        )

        # Add IDs
        for i, example in enumerate(examples):

            example["id"] = str(i)

        splits[split_name] = Dataset.from_list(
            examples
        )

    masakha_datasets[lang] = DatasetDict(
        splits
    )

    print(
        f"  Train:      {len(splits['train'])}"
    )

    print(
        f"  Validation: {len(splits['validation'])}"
    )

    print(
        f"  Test:       {len(splits['test'])}"
    )


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MASAKHANER 1.0 LOADING COMPLETE")
print("=" * 70)

for lang in masakha_langs:

    dataset = masakha_datasets[lang]

    print(
        f"{lang}: "
        f"{len(dataset['train'])} train | "
        f"{len(dataset['validation'])} validation | "
        f"{len(dataset['test'])} test"
    )


Loading MasakhaNER 1.0: amh
  Train:      1750
  Validation: 250
  Test:       500

Loading MasakhaNER 1.0: hau
  Train:      1912
  Validation: 276
  Test:       552

Loading MasakhaNER 1.0: ibo
  Train:      2235
  Validation: 320
  Test:       638

Loading MasakhaNER 1.0: kin
  Train:      2116
  Validation: 302
  Test:       605

Loading MasakhaNER 1.0: lug
  Train:      1428
  Validation: 200
  Test:       407

Loading MasakhaNER 1.0: luo
  Train:      644
  Validation: 92
  Test:       186

Loading MasakhaNER 1.0: pcm
  Train:      2124
  Validation: 306
  Test:       600

Loading MasakhaNER 1.0: swa
  Train:      2109
  Validation: 300
  Test:       604

Loading MasakhaNER 1.0: wol
  Train:      1871
  Validation: 267
  Test:       539

Loading MasakhaNER 1.0: yor
  Train:      2171
  Validation: 305
  Test:       645

MASAKHANER 1.0 LOADING COMPLETE
amh: 1750 train | 250 validation | 500 test
hau: 1912 train | 276 validation | 552 test
ibo: 2235 train | 320 validation | 638 te

# Load Supplemental Amharic

In [ ]:
# ============================================================
# CELL 4 — Supplemental Amharic NER Dataset
# ============================================================

from datasets import load_dataset

print("Loading supplemental Amharic...")

supp_amharic = load_dataset(
    "rasyosef/amharic-named-entity-recognition"
)

print("\nSupplemental Amharic loaded successfully.")

print("=" * 70)
print("SUPPLEMENTAL AMHARIC DATASET")
print("=" * 70)

for split in supp_amharic:
    print(f"{split}: {len(supp_amharic[split])} examples")

print("\nColumns:")
print(supp_amharic["train"].column_names)

print("\nFeatures:")
print(supp_amharic["train"].features)

print("\nFirst example:")
print(supp_amharic["train"][0])

Loading supplemental Amharic...

Supplemental Amharic loaded successfully.
SUPPLEMENTAL AMHARIC DATASET
train: 3465 examples

Columns:
['tokens', 'ner_tags']

Features:
{'tokens': List(Value('string')), 'ner_tags': List(ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-TIME', 'I-TIME', 'B-TTL', 'I-TTL']))}

First example:
{'tokens': ['ኢዴፓ', 'በየክልሉ', 'በሚንቀሳቀስበት', 'ጊዜ', 'ሁሉ', 'የሀገሪቱን', 'አጠቃላይ', 'ሕግእንዲሁም', 'የአካባቢውን', 'ባህልና', 'ቋንቋ', 'አክብሮ', 'በአካባቢው', 'የሚገኙ', 'የፖለቲካ', 'ድርጅቶችንም', 'አክብሮና', 'መብታቸውን', 'ጠብቆ', 'በጨዋነት', 'ያስተምራል', '፣', 'ይማራል"', 'ብለዋል', '።'], 'ner_tags': [3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}


# Inspect NER Label Schemas

In [ ]:
# ============================================================
# Inspect NER Label Schemas
# ============================================================

print("=" * 70)
print("NER LABEL SCHEMAS")
print("=" * 70)


# ------------------------------------------------------------
# Common label mapping used by our MasakhaNER 1.0 loader
# ------------------------------------------------------------

MASAKHA_LABELS = [
    "O",
    "B-PER",
    "I-PER",
    "B-ORG",
    "I-ORG",
    "B-LOC",
    "I-LOC",
    "B-DATE",
    "I-DATE"
]

print("\nMasakhaNER 1.0 label mapping:")
for i, label in enumerate(MASAKHA_LABELS):
    print(f"  {i}: {label}")


# ------------------------------------------------------------
# 1. CoNLL-2003
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("[1] CoNLL-2003")
print("=" * 70)

conll_feature = conll["train"].features["ner_tags"]

print("\nFeature:")
print(conll_feature)

print("\nLabel names:")
print(conll_feature.feature.names)


# ------------------------------------------------------------
# 2. MasakhaNER 1.0 — Amharic
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("[2] MasakhaNER 1.0 — Amharic")
print("=" * 70)

masakha_feature = masakha_datasets["amh"]["train"].features["ner_tags"]

print("\nFeature:")
print(masakha_feature)

print("\nActual labels used:")
print(MASAKHA_LABELS)


# Check actual values
unique_masakha_tags = set()

for row in masakha_datasets["amh"]["train"]:
    unique_masakha_tags.update(row["ner_tags"])

print("\nInteger labels actually present:")
print(sorted(unique_masakha_tags))

print("\nInteger → label:")

for tag_id in sorted(unique_masakha_tags):
    print(f"  {tag_id}: {MASAKHA_LABELS[tag_id]}")


# ------------------------------------------------------------
# 3. Supplemental Amharic
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("[3] Supplemental Amharic")
print("=" * 70)

supp_feature = supp_amharic["train"].features["ner_tags"]

print("\nFeature:")
print(supp_feature)

print("\nSupplemental label names:")

if hasattr(supp_feature.feature, "names"):
    print(supp_feature.feature.names)
else:
    print("Supplemental dataset does not expose ClassLabel metadata.")


# ------------------------------------------------------------
# 4. Column structures
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("COLUMN STRUCTURE")
print("=" * 70)

print("\nCoNLL:")
print(conll["train"].column_names)

print("\nMasakhaNER Amharic:")
print(masakha_datasets["amh"]["train"].column_names)

print("\nSupplemental Amharic:")
print(supp_amharic["train"].column_names)

NER LABEL SCHEMAS

MasakhaNER 1.0 label mapping:
  0: O
  1: B-PER
  2: I-PER
  3: B-ORG
  4: I-ORG
  5: B-LOC
  6: I-LOC
  7: B-DATE
  8: I-DATE

[1] CoNLL-2003

Feature:
List(ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']))

Label names:
['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

[2] MasakhaNER 1.0 — Amharic

Feature:
List(Value('int64'))

Actual labels used:
['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-DATE', 'I-DATE']

Integer labels actually present:
[0, 1, 2, 3, 4, 5, 6, 7, 8]

Integer → label:
  0: O
  1: B-PER
  2: I-PER
  3: B-ORG
  4: I-ORG
  5: B-LOC
  6: I-LOC
  7: B-DATE
  8: I-DATE

[3] Supplemental Amharic

Feature:
List(ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-TIME', 'I-TIME', 'B-TTL', 'I-TTL']))

Supplemental label names:
['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-TIME', 'I-TIME', 'B-TTL', 'I-TTL']

COLUMN 

In [ ]:
# ============================================================
# Deduplicate & Merge Amharic Data
# ============================================================

import hashlib
from datasets import Value, Sequence
from datasets import concatenate_datasets


# ============================================================
# 1. Build leakage-prevention hash set from validation/test
# ============================================================

amh_val_test_texts = set()

for split in ["validation", "test"]:

    for row in masakha_datasets["amh"][split]:

        text = " ".join(row["tokens"])

        amh_val_test_texts.add(
            hashlib.md5(
                text.encode("utf-8")
            ).hexdigest()
        )

print(
    f"MasakhaNER Amharic validation/test hashes: "
    f"{len(amh_val_test_texts)}"
)


# ============================================================
# 2. Remove leakage from supplemental Amharic
# ============================================================

def is_not_leakage(row):

    h = hashlib.md5(
        " ".join(row["tokens"]).encode("utf-8")
    ).hexdigest()

    return h not in amh_val_test_texts


filtered_supp = supp_amharic["train"].filter(
    is_not_leakage
)

print(
    f"Supplemental Amharic before deduplication: "
    f"{len(supp_amharic['train'])}"
)

print(
    f"Supplemental Amharic after deduplication:  "
    f"{len(filtered_supp)}"
)

print(
    f"Removed: "
    f"{len(supp_amharic['train']) - len(filtered_supp)}"
)


# ============================================================
# 3. Collapse TIME and TTL → O
# ============================================================

def remap_supp_tags(row):

    row["ner_tags"] = [
        0 if tag >= 7 else tag
        for tag in row["ner_tags"]
    ]

    return row


filtered_supp = filtered_supp.map(
    remap_supp_tags
)


# ============================================================
# 4. IMPORTANT:
#    Convert supplemental ner_tags to List(Value("int64"))
#
#    MasakhaNER 1.0 was loaded with:
#
#        List(Value("int64"))
#
#    So we must use the exact same feature type.
# ============================================================

int_tag_feature = Sequence(
    Value("int64")
)

filtered_supp = filtered_supp.cast_column(
    "ner_tags",
    int_tag_feature
)


# ============================================================
# 5. Add synthetic ID
# ============================================================

filtered_supp = filtered_supp.map(
    lambda row, idx: {
        "id": str(idx)
    },
    with_indices=True
)


# ============================================================
# 6. Add language
# ============================================================

filtered_supp = filtered_supp.map(
    lambda row: {
        "language": "amh"
    }
)


# ============================================================
# 7. Select columns
# ============================================================

filtered_supp = filtered_supp.select_columns(
    [
        "id",
        "tokens",
        "ner_tags",
        "language"
    ]
)


# ============================================================
# 8. Prepare MasakhaNER Amharic TRAIN
# ============================================================

amh_train = masakha_datasets["amh"]["train"]

amh_train = amh_train.map(
    lambda row: {
        "language": "amh"
    }
)

amh_train = amh_train.select_columns(
    [
        "id",
        "tokens",
        "ner_tags",
        "language"
    ]
)


# ============================================================
# 9. Make absolutely sure MasakhaNER has int64 tags
# ============================================================

amh_train = amh_train.cast_column(
    "ner_tags",
    int_tag_feature
)


# ============================================================
# 10. Verify feature compatibility BEFORE merging
# ============================================================

print("\n" + "=" * 70)
print("FEATURE CHECK")
print("=" * 70)

print("\nMasakhaNER:")
print(amh_train.features)

print("\nSupplemental:")
print(filtered_supp.features)


# ============================================================
# 11. Merge
# ============================================================

merged_amh_train = concatenate_datasets(
    [
        amh_train,
        filtered_supp
    ]
)


# ============================================================
# 12. Replace Amharic TRAIN split
# ============================================================

masakha_datasets["amh"]["train"] = merged_amh_train


# ============================================================
# 13. Final summary
# ============================================================

print("\n" + "=" * 70)
print("AMHARIC MERGE COMPLETE")
print("=" * 70)

print(
    f"Original MasakhaNER train: {len(amh_train)}"
)

print(
    f"Supplemental kept: {len(filtered_supp)}"
)

print(
    f"Merged Amharic train: "
    f"{len(masakha_datasets['amh']['train'])}"
)

print(
    f"Validation: "
    f"{len(masakha_datasets['amh']['validation'])}"
)

print(
    f"Test: "
    f"{len(masakha_datasets['amh']['test'])}"
)

MasakhaNER Amharic validation/test hashes: 750
Supplemental Amharic before deduplication: 3465
Supplemental Amharic after deduplication:  3465
Removed: 0


Map:   0%|          | 0/1750 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/1750 [00:00<?, ? examples/s]


FEATURE CHECK

MasakhaNER:
{'id': Value('string'), 'tokens': List(Value('string')), 'ner_tags': List(Value('int64')), 'language': Value('string')}

Supplemental:
{'id': Value('string'), 'tokens': List(Value('string')), 'ner_tags': List(Value('int64')), 'language': Value('string')}

AMHARIC MERGE COMPLETE
Original MasakhaNER train: 1750
Supplemental kept: 3465
Merged Amharic train: 5215
Validation: 250
Test: 500


In [ ]:
# ============================================================
# Build Combined Multilingual Dataset
# ============================================================

from datasets import ClassLabel, Sequence, DatasetDict, concatenate_datasets


# ============================================================
# 1. Final unified label schema
# ============================================================

label_list = [
    "O",
    "B-PER",
    "I-PER",
    "B-ORG",
    "I-ORG",
    "B-LOC",
    "I-LOC"
]

unified_feature = Sequence(
    ClassLabel(names=label_list)
)


# ============================================================
# 2. Add language column
# ============================================================

def add_lang_column(dataset, lang_code):

    return dataset.map(
        lambda x: {"language": lang_code}
    )


# English
conll = add_lang_column(
    conll,
    "eng"
)


# African languages
for lang in masakha_langs:

    masakha_datasets[lang] = add_lang_column(
        masakha_datasets[lang],
        lang
    )


# ============================================================
# 3. Unify tags
# ============================================================
#
# Anything >= 7 becomes O.
#
# CoNLL:
#   MISC -> O
#
# MasakhaNER:
#   DATE -> O
#
# Supplemental Amharic:
#   TIME / TTL -> O
#
# ============================================================

def unify_tags(example):

    example["ner_tags"] = [
        0 if tag >= 7 else tag
        for tag in example["ner_tags"]
    ]

    return example


conll = conll.map(unify_tags)

for lang in masakha_langs:

    masakha_datasets[lang] = masakha_datasets[lang].map(
        unify_tags
    )


# ============================================================
# 4. CAST EVERYTHING TO THE SAME FEATURE
# ============================================================

print("Casting datasets to unified 7-label schema...")


# CoNLL
conll = conll.cast_column(
    "ner_tags",
    unified_feature
)


# MasakhaNER
for lang in masakha_langs:

    masakha_datasets[lang] = masakha_datasets[lang].cast_column(
        "ner_tags",
        unified_feature
    )


# ============================================================
# 5. Select common columns
# ============================================================

columns_to_keep = [
    "id",
    "tokens",
    "ner_tags",
    "language"
]


conll = conll.select_columns(
    columns_to_keep
)


for lang in masakha_langs:

    masakha_datasets[lang] = masakha_datasets[lang].select_columns(
        columns_to_keep
    )


# ============================================================
# 6. Build train / validation / test lists
# ============================================================

train_sets = (
    [conll["train"]]
    +
    [
        masakha_datasets[lang]["train"]
        for lang in masakha_langs
    ]
)


val_sets = (
    [conll["validation"]]
    +
    [
        masakha_datasets[lang]["validation"]
        for lang in masakha_langs
    ]
)


test_sets = (
    [conll["test"]]
    +
    [
        masakha_datasets[lang]["test"]
        for lang in masakha_langs
    ]
)


# ============================================================
# 7. Concatenate
# ============================================================

combined_train = concatenate_datasets(
    train_sets
)

combined_validation = concatenate_datasets(
    val_sets
)

combined_test = concatenate_datasets(
    test_sets
)


# ============================================================
# 8. Shuffle TRAIN only
# ============================================================

combined_train = combined_train.shuffle(
    seed=42
)


# ============================================================
# 9. Create DatasetDict
# ============================================================

combined_dataset = DatasetDict({

    "train": combined_train,

    "validation": combined_validation,

    "test": combined_test
})


# ============================================================
# 10. Print summary
# ============================================================

print("\n" + "=" * 70)
print("COMBINED DATASET")
print("=" * 70)

print(
    f"Train:      {len(combined_dataset['train'])}"
)

print(
    f"Validation: {len(combined_dataset['validation'])}"
)

print(
    f"Test:       {len(combined_dataset['test'])}"
)


# ============================================================
# 11. Language distribution
# ============================================================

print("\n" + "=" * 70)
print("TRAINING EXAMPLES BY LANGUAGE")
print("=" * 70)

all_langs = ["eng"] + masakha_langs

for lang in all_langs:

    count = sum(
        1
        for row in combined_dataset["train"]
        if row["language"] == lang
    )

    print(
        f"{lang:>4}: {count}"
    )

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

Map:   0%|          | 0/5215 [00:00<?, ? examples/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

...

Casting the dataset:   0%|          | 0/539 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/2171 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/305 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/645 [00:00<?, ? examples/s]


COMBINED DATASET
Train:      35866
Validation: 5868
Test:       8729

TRAINING EXAMPLES BY LANGUAGE
 eng: 14041
 amh: 5215
 hau: 1912
 ibo: 2235
 kin: 2116
 lug: 1428
 luo: 644
 pcm: 2124
 swa: 2109
 wol: 1871
 yor: 2171


In [ ]:
# ============================================================
# CELL 8 — Tokenize & Align BIO Labels
# ============================================================

def tokenize_and_align_labels(examples):

    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=128,
        padding=False
    )

    labels = []

    for i, label in enumerate(examples["ner_tags"]):

        word_ids = tokenized_inputs.word_ids(
            batch_index=i
        )

        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:

            # Special tokens
            if word_idx is None:

                label_ids.append(-100)

            # First subword of a word
            elif word_idx != previous_word_idx:

                label_ids.append(
                    label[word_idx]
                )

            # Additional subwords
            else:

                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels

    return tokenized_inputs


# ============================================================
# Apply tokenization
# ============================================================

print("Tokenizing training data...")

tokenized_datasets = combined_dataset.map(
    tokenize_and_align_labels,
    batched=True
)


# ============================================================
# Check results
# ============================================================

print("\n" + "=" * 70)
print("TOKENIZATION COMPLETE")
print("=" * 70)

print(
    "Train:",
    len(tokenized_datasets["train"])
)

print(
    "Validation:",
    len(tokenized_datasets["validation"])
)

print(
    "Test:",
    len(tokenized_datasets["test"])
)


# ============================================================
# Inspect one example
# ============================================================

example = tokenized_datasets["train"][0]

print("\nExample:")
print("Tokens:", example["tokens"][:20])
print("Labels:", example["ner_tags"][:20])
print("Input IDs:", example["input_ids"][:20])
print("Aligned labels:", example["labels"][:20])

Tokenizing training data...


Map:   0%|          | 0/35866 [00:00<?, ? examples/s]

Map:   0%|          | 0/5868 [00:00<?, ? examples/s]

Map:   0%|          | 0/8729 [00:00<?, ? examples/s]


TOKENIZATION COMPLETE
Train: 35866
Validation: 5868
Test: 8729

Example:
Tokens: ['CSKA', 'Moscow', '25', '13', '6', '6', '40', '27', '45']
Labels: [3, 4, 0, 0, 0, 0, 0, 0, 0]
Input IDs: [0, 313, 28209, 124338, 714, 702, 305, 305, 1112, 1438, 2678, 2]
Aligned labels: [-100, 3, -100, 4, 0, 0, 0, 0, 0, 0, 0, -100]


In [ ]:
# ============================================================
# Configure XLM-R for Token Classification
# ============================================================

import torch
from transformers import AutoModelForTokenClassification

MODEL_CHECKPOINT = "xlm-roberta-base"

# Final NER labels
label_list = [
    "O",
    "B-PER",
    "I-PER",
    "B-ORG",
    "I-ORG",
    "B-LOC",
    "I-LOC"
]

id2label = {
    i: label
    for i, label in enumerate(label_list)
}

label2id = {
    label: i
    for i, label in enumerate(label_list)
}

# Check GPU
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 70)
print("MODEL CONFIGURATION")
print("=" * 70)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        f"GPU memory: "
        f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB"
    )
else:
    print("WARNING: CUDA is not available.")


# ============================================================
# Load XLM-R
# ============================================================

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

model.to(device)


print("\nModel loaded successfully.")
print("Checkpoint:", MODEL_CHECKPOINT)
print("Number of labels:", len(label_list))
print("Labels:", label_list)

MODEL CONFIGURATION
Device: cuda
GPU: Tesla T4
GPU memory: 15.64 GB


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Model loaded successfully.
Checkpoint: xlm-roberta-base
Number of labels: 7
Labels: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']


In [ ]:
# ============================================================
# Training Configuration
# ============================================================

from transformers import TrainingArguments, DataCollatorForTokenClassification

# ------------------------------------------------------------
# GPU check
# ------------------------------------------------------------

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print("CUDA available: True")
else:
    print("WARNING: CUDA is not available.")
    print("Training will run on CPU and may be very slow.")


# ------------------------------------------------------------
# Training configuration
# ------------------------------------------------------------

OUTPUT_DIR = "./models/ner-multilingual-model"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Evaluation and checkpointing
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    # Optimization
    learning_rate=2e-5,
    weight_decay=0.01,

    # Batch sizes
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    # Training duration
    num_train_epochs=3,

    # Logging
    logging_steps=100,
    logging_strategy="steps",

    # Select the best checkpoint
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    # Reproducibility
    seed=42,

    # No external experiment tracker
    report_to="none",

    # Mixed precision on NVIDIA GPU
    fp16=torch.cuda.is_available(),
)


# ------------------------------------------------------------
# Data collator
# ------------------------------------------------------------

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)


print("\n" + "=" * 70)
print("TRAINING CONFIGURATION")
print("=" * 70)

print("Output directory:", OUTPUT_DIR)
print("Learning rate:", training_args.learning_rate)
print("Train batch size:", training_args.per_device_train_batch_size)
print("Evaluation batch size:", training_args.per_device_eval_batch_size)
print("Epochs:", training_args.num_train_epochs)
print("Weight decay:", training_args.weight_decay)
print("FP16:", training_args.fp16)
print("Evaluation:", training_args.eval_strategy)
print("Checkpoint saving:", training_args.save_strategy)

GPU: Tesla T4
CUDA available: True

TRAINING CONFIGURATION
Output directory: ./models/ner-multilingual-model
Learning rate: 2e-05
Train batch size: 16
Evaluation batch size: 16
Epochs: 3
Weight decay: 0.01
FP16: True
Evaluation: IntervalStrategy.EPOCH
Checkpoint saving: SaveStrategy.EPOCH


In [ ]:
# ============================================================
# Evaluation Metrics + Trainer
# ============================================================
from transformers import Trainer
seqeval_metric = evaluate.load("seqeval")


def align_predictions(predictions, labels):

    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [
            label_list[p]
            for p, l in zip(prediction, label)
            if l != -100
        ]
        for prediction, label in zip(predictions, labels)
    ]

    true_labels = [
        [
            label_list[l]
            for p, l in zip(prediction, label)
            if l != -100
        ]
        for prediction, label in zip(predictions, labels)
    ]

    return true_predictions, true_labels


def compute_metrics(p):

    predictions, labels = p

    true_predictions, true_labels = align_predictions(
        predictions,
        labels
    )

    results = seqeval_metric.compute(
        predictions=true_predictions,
        references=true_labels
    )

    # All-O collapse check
    all_pred_tags = [
        tag
        for sequence in true_predictions
        for tag in sequence
    ]

    non_o_ratio = (
        sum(tag != "O" for tag in all_pred_tags)
        / max(len(all_pred_tags), 1)
    )

    if non_o_ratio < 0.01:
        print(
            "\nWARNING: Model is predicting almost entirely O."
        )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }


# ============================================================
# Create Trainer
# ============================================================

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],

    data_collator=data_collator,

    compute_metrics=compute_metrics
)


print("=" * 70)
print("TRAINER READY")
print("=" * 70)

print(
    "Training examples:",
    len(tokenized_datasets["train"])
)

print(
    "Validation examples:",
    len(tokenized_datasets["validation"])
)

print(
    "Number of labels:",
    len(label_list)
)

print("Ready to start training.")

TRAINER READY
Training examples: 35866
Validation examples: 5868
Number of labels: 7
Ready to start training.


In [ ]:
import numpy as np

In [ ]:
# ============================================================
# Start Training
# ============================================================

print("=" * 70)
print("STARTING TRAINING")
print("=" * 70)

print("Device:", device)
print("Training examples:", len(tokenized_datasets["train"]))
print("Validation examples:", len(tokenized_datasets["validation"]))
print("Epochs:", training_args.num_train_epochs)

trainer.train()

STARTING TRAINING
Device: cuda
Training examples: 35866
Validation examples: 5868
Epochs: 3


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.076100,0.057226,0.885402,0.906150,0.895656,0.982930
2,0.057538,0.054206,0.903549,0.916667,0.910061,0.984695
3,0.034609,0.055975,0.901862,0.919182,0.910439,0.984796


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6726, training_loss=0.07500532959814125, metrics={'train_runtime': 1055.7315, 'train_samples_per_second': 101.918, 'train_steps_per_second': 6.371, 'total_flos': 5412773249371356.0, 'train_loss': 0.07500532959814125, 'epoch': 3.0})

In [ ]:
# ============================================================
# SAVE FINAL MODEL TO GOOGLE DRIVE
# ============================================================

FINAL_MODEL_DIR = "/content/drive/MyDrive/NER_Project/ner-multilingual-model/final"

os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print("=" * 70)
print("MODEL SAVED")
print("=" * 70)

print("Location:")
print(FINAL_MODEL_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

MODEL SAVED
Location:
/content/drive/MyDrive/NER_Project/ner-multilingual-model/final


In [ ]:
# ============================================================
# VERIFY SAVED MODEL
# ============================================================

print("=" * 70)
print("SAVED MODEL FILES")
print("=" * 70)

for root, dirs, files in os.walk(FINAL_MODEL_DIR):
    print(f"\n{root}")
    for file in files:
        print("  ", file)

SAVED MODEL FILES

/content/drive/MyDrive/NER_Project/ner-multilingual-model/final
   config.json
   model.safetensors
   tokenizer_config.json
   tokenizer.json
   training_args.bin


In [ ]:
# ============================================================
# FINAL TEST EVALUATION
# ============================================================

print("=" * 70)
print("FINAL TEST EVALUATION")
print("=" * 70)

test_output = trainer.predict(
    tokenized_datasets["test"]
)

test_predictions, test_labels = align_predictions(
    test_output.predictions,
    test_output.label_ids
)

test_results = seqeval_metric.compute(
    predictions=test_predictions,
    references=test_labels
)

print("\nFINAL TEST RESULTS")
print("-" * 70)

print(f"Precision : {test_results['overall_precision']:.4f}")
print(f"Recall    : {test_results['overall_recall']:.4f}")
print(f"F1        : {test_results['overall_f1']:.4f}")
print(f"Accuracy  : {test_results['overall_accuracy']:.4f}")

FINAL TEST EVALUATION



FINAL TEST RESULTS
----------------------------------------------------------------------
Precision : 0.8402
Recall    : 0.8703
F1        : 0.8550
Accuracy  : 0.9772


In [ ]:
import pandas as pd

In [ ]:
# ============================================================
# PER-LANGUAGE TEST EVALUATION
# ============================================================

all_langs = ["eng"] + masakha_langs

language_results = []

for lang in all_langs:

    print(f"Evaluating {lang}...")

    lang_subset = tokenized_datasets["test"].filter(
        lambda x: x["language"] == lang
    )

    if len(lang_subset) == 0:
        continue

    output = trainer.predict(lang_subset)

    predictions, labels = align_predictions(
        output.predictions,
        output.label_ids
    )

    results = seqeval_metric.compute(
        predictions=predictions,
        references=labels
    )

    language_results.append({
        "language": lang,
        "sentences": len(lang_subset),
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"]
    })

per_language_df = pd.DataFrame(language_results)

per_language_df = per_language_df.sort_values(
    "f1",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 70)
print("PER-LANGUAGE RESULTS")
print("=" * 70)

print(per_language_df.to_string(index=False))

Evaluating eng...


Evaluating amh...


Evaluating hau...


Evaluating ibo...


Evaluating kin...


...

Evaluating pcm...


Evaluating swa...


Evaluating wol...


Evaluating yor...



PER-LANGUAGE RESULTS
language  sentences  precision   recall       f1
     eng       3453   0.916232 0.925506 0.920846
     hau        552   0.896484 0.935780 0.915711
     swa        604   0.874046 0.900688 0.887167
     ibo        638   0.849237 0.862403 0.855769
     pcm        600   0.842593 0.866667 0.854460
     yor        645   0.760823 0.826087 0.792113
     lug        407   0.759740 0.791610 0.775348
     kin        605   0.685300 0.789976 0.733925
     luo        186   0.724852 0.729167 0.727003
     amh        500   0.704444 0.701327 0.702882
     wol        539   0.636175 0.706697 0.669584


In [ ]:
# ============================================================
# CELL — MACRO F1
# ============================================================

macro_f1 = per_language_df["f1"].mean()

print("=" * 70)
print("MACRO F1")
print("=" * 70)

print(
    f"Macro F1 across {len(per_language_df)} languages: "
    f"{macro_f1:.4f}"
)

print(
    f"Overall pooled F1: "
    f"{test_results['overall_f1']:.4f}"
)

MACRO F1
Macro F1 across 11 languages: 0.8032
Overall pooled F1: 0.8550


In [ ]:
# ============================================================
# CELL — WORST PERFORMING LANGUAGE
# ============================================================

worst_language = per_language_df.iloc[-1]

print("=" * 70)
print("WORST-PERFORMING LANGUAGE")
print("=" * 70)

print(
    f"Language : {worst_language['language']}"
)

print(
    f"Sentences: {worst_language['sentences']}"
)

print(
    f"Precision: {worst_language['precision']:.4f}"
)

print(
    f"Recall   : {worst_language['recall']:.4f}"
)

print(
    f"F1       : {worst_language['f1']:.4f}"
)

WORST-PERFORMING LANGUAGE
Language : wol
Sentences: 539
Precision: 0.6362
Recall   : 0.7067
F1       : 0.6696


In [ ]:
# ============================================================
# MANUAL ERROR ANALYSIS
# ============================================================

worst_lang = worst_language["language"]

worst_subset = tokenized_datasets["test"].filter(
    lambda x: x["language"] == worst_lang
)

worst_output = trainer.predict(worst_subset)

worst_predictions, worst_labels = align_predictions(
    worst_output.predictions,
    worst_output.label_ids
)

print("=" * 70)
print(f"ERROR ANALYSIS — {worst_lang}")
print("=" * 70)

n_show = min(10, len(worst_subset))

for i in range(n_show):

    print("\nTokens:")
    print(worst_subset[i]["tokens"])

    print("\nGold:")
    print(worst_labels[i])

    print("\nPredicted:")
    print(worst_predictions[i])

    print("-" * 70)

Filter:   0%|          | 0/8729 [00:00<?, ? examples/s]

ERROR ANALYSIS — wol

Tokens:
['Bu', 'nu', 'delloo', 'woon', 'waat', 'yi', 'farañse', 'àbb', 'seen', 'i', 'woroom', ',', 'lu', 'farañsey', 'dese', 'ciy', 'waat', '?', '😅', 'Ndax', 'farañse', 'dina', 'amati', '?', '😫', 'Ndax', 'farañse', 'bi', 'dina', 'manati', 'wax', 'soxlaam', '?', '🥺']

Gold:
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

Predicted:
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
----------------------------------------------------------------------

Tokens:
['Bokk', 'yi', ',', 'tontuleen', 'laaj', 'yi', '!']

Gold:
['O', 'O', 'O', 'O', 'O', 'O', 'O']

Predicted:
['O', 'O', 'O', 'O', 'O', 'O', 'O']
----------------------------------------------------------------------

Tokens:
['!', '🤝', '🏿', '👊', '🏿']

Gold:
['O', 'O', 'O'

In [ ]:
# ============================================================
# RELOAD SAVED MODEL
# ============================================================

from transformers import AutoModelForTokenClassification

saved_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_DIR
)

saved_tokenizer = AutoTokenizer.from_pretrained(
    FINAL_MODEL_DIR,
    use_fast=True
)

print("=" * 70)
print("SAVED MODEL RELOADED SUCCESSFULLY")
print("=" * 70)

print("Model:", type(saved_model).__name__)
print("Tokenizer:", type(saved_tokenizer).__name__)

ValueError: Unrecognized model in /content/drive/MyDrive/NER_Project/ner-multilingual-model. Should have a `model_type` key in its config.json.

In [ ]:
import os

print("Files in /content:")
for item in os.listdir("/content"):
    print(item)

Files in /content:
.config
models
drive
sample_data


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

print("\nFiles in Google Drive:")
for item in os.listdir("/content/drive/MyDrive"):
    print(item)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Files in Google Drive:
Alx
Google Workspace Skills Checklist_ PF.gsheet
Untitled document (33).gdoc
500 useful words.gdoc
Untitled document (32).gdoc
Untitled document (31).gdoc
Solomon’s slide 1 (1).gslides
4_5771610235282135923.docx
Following a Story Arc_ Into Another World-1.pdf
Problem impact assessment.gdoc
How to make an interview .gdoc
Week 4 Milestone Worksheet - Professional Foundations.gdoc
Professional Foundations - Week 3 Milestone Rubric.gdoc
PICS and Personal Mission Statement Worksheet - ALX Foundations.gdoc
Untitled document (30).gdoc
Week 3 Milestone Worksheet - Professional Foundations.gdoc
Untitled spreadsheet (3).gsheet
Copy of Professional Foundations - Week 5 Milestone Rubric.gdoc
Copy of Professional Foundations - Week 4 Milestone Rubric.gdoc
Copy of Week 4 Milestone Worksheet - Professional Foundations (2).gdoc
Copy of Week 4 Mileston

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

MODEL_PATH = "/content/"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model = AutoModelForTokenClassification.from_pretrained(MODEL_PATH)

print("Model reloaded successfully.")
print("Number of labels:", model.config.num_labels)

OSError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/content/ner_model'. Use `repo_type` argument if needed.

In [ ]:
from transformers import pipeline
ner_pipeline = pipeline(
    "ner",
    model=saved_model,
    tokenizer=saved_tokenizer,
    aggregation_strategy="simple"
)

print("NER pipeline ready.")

In [ ]:
# ============================================================
# PREDICTION FUNCTION
# ============================================================

def predict_entities(text):

    results = ner_pipeline(text)

    return [
        {
            "entity": result["entity_group"],
            "word": result["word"],
            "score": round(float(result["score"]), 4)
        }
        for result in results
    ]

In [ ]:
# ============================================================
# INFERENCE SMOKE TEST
# ============================================================


examples = [
    "Nelson Mandela was born in South Africa.",
    "Barack Obama was the president of the United States.",
]

for text in examples:

    print("\nText:")
    print(text)

    print("\nEntities:")

    for entity in predict_entities(text):
        print(entity)

    print("-" * 70)

NameError: name 'pipeline' is not defined

In [ ]:
# ============================================================
# CELL — CREATE MODEL BACKUP
# ============================================================

import shutil

backup_path = "/content/ner-multilingual-model-final"

shutil.make_archive(
    backup_path,
    "zip",
    FINAL_MODEL_DIR
)

print("Backup created:")
print(backup_path + ".zip")

# Model uploading

In [1]:
!pip install -U huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.2/793.2 kB 14.2 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.27.0
    Uninstalling huggingface_hub-1.27.0:
      Successfully uninstalled huggingface_hub-1.27.0


In [2]:
from huggingface_hub import login

login()

In [6]:
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [7]:
from huggingface_hub import HfApi

api = HfApi()

api.upload_folder(
    folder_path="/content/drive/MyDrive/NER_Project/ner-multilingual-model/final",
    repo_id="SoloCode/multilingual-ner",
    repo_type="model"
)

CommitInfo(commit_url='https://huggingface.co/SoloCode/multilingual-ner/commit/c4dfa77dcb22dcc1a67819183ded5eb2c3464412', commit_message='Upload folder using huggingface_hub', commit_description='', oid='c4dfa77dcb22dcc1a67819183ded5eb2c3464412', pr_url=None, repo_url=RepoUrl('https://huggingface.co/SoloCode/multilingual-ner', endpoint='https://huggingface.co', repo_type='model', repo_id='SoloCode/multilingual-ner'), pr_revision=None, pr_num=None)

In [9]:
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification
)

MODEL_ID = "SoloCode/multilingual-ner"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_ID
)

print("Model loaded successfully!")
print(model.config.id2label)

config.json:   0%|          | 0.00/1.03k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/343 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model loaded successfully!
{0: 'O', 1: 'B-PER', 2: 'I-PER', 3: 'B-ORG', 4: 'I-ORG', 5: 'B-LOC', 6: 'I-LOC'}


In [11]:
from transformers import pipeline
ner_pipeline = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)

print("NER pipeline ready.")

NER pipeline ready.


In [12]:
test_sentences = [
    "Nelson Mandela was born in South Africa.",
    "Abiy Ahmed is the Prime Minister of Ethiopia.",
    "አብይ አህመድ የኢትዮጵያ ጠቅላይ ሚኒስትር ነው።"
]

for text in test_sentences:

    print("\n" + "=" * 70)
    print("TEXT:", text)
    print("=" * 70)

    results = ner_pipeline(text)

    for entity in results:
        print(
            f"{entity['word']:25} "
            f"{entity['entity_group']:10} "
            f"{entity['score']:.4f}"
        )


TEXT: Nelson Mandela was born in South Africa.
Nelson Mandela            PER        0.9986
South Africa              LOC        0.9984

TEXT: Abiy Ahmed is the Prime Minister of Ethiopia.
Abi                       PER        0.9990
y Ahmed                   PER        0.9132
Ethiopia                  LOC        0.9988

TEXT: አብይ አህመድ የኢትዮጵያ ጠቅላይ ሚኒስትር ነው።
አብይ አህመድ                  PER        0.9925


# Summary

This notebook implemented an end-to-end pipeline for developing a **multilingual Named Entity Recognition model for KYC-oriented text**.

The major stages of the project were:

1. **Dataset Exploration**  
   The available annotated datasets were examined to understand their structure, entity categories, annotation formats, and data distributions.

2. **Data Preparation**  
   The datasets were cleaned and transformed into a consistent format suitable for supervised token classification.

3. **Label Standardization**  
   Differences between dataset-specific entity labels were addressed so that semantically equivalent entities could be represented consistently.

4. **Dataset Merging**  
   Compatible datasets were combined to increase the diversity and coverage of the training data. This provides the model with exposure to a broader range of linguistic and entity-level patterns.

5. **BIO Representation**  
   Entity annotations were represented using the BIO scheme, allowing entity boundaries to be explicitly represented at the token level.

6. **Tokenization and Label Alignment**  
   The text was tokenized using the selected multilingual transformer tokenizer, while preserving the correspondence between tokens and their entity labels.

7. **Model Fine-Tuning**  
   A pretrained multilingual transformer was fine-tuned as a token-classification model for the NER task.

8. **Evaluation**  
   The trained model was evaluated using NER evaluation metrics to assess its ability to identify and classify entities.

9. **Model Publication**  
   The final trained model was published to the Hugging Face Model Hub:

   **[SoloCode/multilingual-ner](https://huggingface.co/SoloCode/multilingual-ner)**

10. **Application Integration**  
    The model was integrated into a Streamlit application to provide an accessible interface for interactive inference.

    **[Live Streamlit Application](https://6hdjukgmcud7jdh3rtxihf.streamlit.app/)**

## Key Takeaway

The project demonstrates a complete applied NLP workflow in which **annotated multilingual data is transformed into a deployable transformer-based NER system**.

An important lesson from the project is that model performance depends not only on the choice of architecture, but also on the quality and compatibility of the training data. Dataset merging, label consistency, annotation quality, and language representation are therefore fundamental components of the overall system.

At the same time, the resulting NER model should be considered an **information-extraction component rather than a complete KYC verification system**. Real-world deployment would require additional document processing, validation, identity verification, security, privacy, and monitoring mechanisms.

## Future Improvements

Potential future work includes:

- Expanding multilingual KYC-specific training data
- Improving representation of underrepresented languages
- Performing more extensive language-specific evaluation
- Improving annotation consistency
- Addressing class imbalance
- Investigating domain-adaptive pretraining
- Comparing alternative multilingual transformer architectures
- Exploring model compression and quantization
- Integrating OCR and document-layout analysis
- Building a complete document-to-structured-KYC pipeline